# 第 9 章 マクロ経済データの特徴: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/09-macro-data-features.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
smoothing = 1600  # HP フィルターの平滑化の強さ λ（四半期データでは 1600）
series_name = "gdp"  # トレンドを描く系列（gdp・consumption・investment）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and (
    Path(getattr(_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 循環の統計

In [ ]:
import pandas as pd

from lib.chapter09.cycles import cycle_statistics, load_quarterly

df = load_quarterly()
stats = cycle_statistics(df, smoothing)
print("変動の大きさ（標準偏差）")
print(stats["volatility"].round(3).to_string())
print("GDP との相関")
print(stats["correlation_with_gdp"].round(3).to_string())
print("自己相関（ラグ 1）")
print(stats["autocorrelation"].round(3).to_string())

## 図: トレンドと循環

In [ ]:
from lib.chapter09.plot import plot_trend

plot_trend(pd.Series(df[series_name]), smoothing)

In [ ]:
from lib.chapter09.cycles import log_cycle
from lib.chapter09.plot import plot_cycle

plot_cycle(
    pd.DataFrame(
        {
            name: log_cycle(pd.Series(df[name]), smoothing)
            for name in ("gdp", "consumption", "investment")
        }
    )
)

In [ ]:
from lib.chapter09.plot import plot_volatility

plot_volatility(stats)

## 試してみよう

- `smoothing = 100` にすると、GDP の循環の標準偏差は 1.008 から 0.758 に、
  ラグ 1 の自己相関は 0.623 から 0.371 に下がります（トレンドが曲がりやすく、
  循環として残る部分が減ります）
- `smoothing = 100000` にすると、GDP の標準偏差は 1.200、
  自己相関は 0.722 に増えます（トレンドが直線に近づきます）
- どちらでも、投資が最も大きく動き、失業率は GDP と逆向き（相関 −0.94 から −0.97）
  に動く結果は、変わりません